# Uvicorn/JupyUvi helpers

In [ ]:
#| default_exp server

In [ ]:
# await greet('pote.server')

<!-- linkedto: _21a088dd -->
Solveit version: **0.0.85**  
dialoghelper version: **0.2.16**  
git branch: **dev-log**  
git changes: [' M ../dutil/hx_bridge.js', ' M ../explorer/htmx_bridge.ipynb', ' M 00_htmx_bridge.ipynb', '?? ../explorer/.sesskey']

Hi! I'm Vic, we're going to work together on the project **pote.server**.

In [ ]:
#| export
import json, os, asyncio, logging, subprocess, inspect
from contextlib import asynccontextmanager
from datetime import datetime
from functools import partial
from fastcore.basics import first
from fastcore.xml import FT
from fasthtml.core import FastHTML
from fasthtml.jupyter import JupyUvi, JupyUviAsync, is_port_free, HTMX
from fasthtml.xtend import Script, Link
from pote.dutil import set_var

In [ ]:
from itertools import count
from ipykernel_helper import info_md
from fastcore.basics import setify
from fastcore.foundation import store_attr, patch
from fastcore.test import *
from pyskills import doc
from pyskills import *
from fasthtml.components import Div, Span
from dialoghelper.core import add_msg, iife_a, add_html_a, read_msg, add_scr_a, js_run_a, js_eval_a
from pote.logger import logger
from pote.dutil import link_msg, wait_for_a

In [ ]:
if globals['srv']: srv.stop()
srv=app=rt=p = None

In [ ]:
# from inspect import getsource
# await add_msg(getsource(fasthtml.jupyter))

Please see above message with the source code for `fasthtml.jupyter`. What does it do? Explain.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

`fasthtml.jupyter` provides utilities for running and displaying FastHTML/HTMX web apps **inside notebooks** (Jupyter, Solveit, etc.). Here's what the key pieces do:

**Server management**
- `nb_serve` / `nb_serve_async` — start a uvicorn server in a background thread (or async task) so it doesn't block the kernel. Waits until the server is actually ready before returning.
- `JupyUvi` / `JupyUviAsync` — convenient wrapper classes around the above, with `.start()` / `.stop()` lifecycle methods and automatic HTMX port configuration when not in Solveit.
- `wait_port_free` — blocks until a port is released (useful after `.stop()`).

**Display helpers**
- `show` — renders FastHTML components in the notebook, and triggers `htmx.process()` so htmx attributes on dynamically injected HTML are activated.
- `render_ft` — patches `FT._repr_html_` so FastHTML components auto-render with HTMX support when returned from a cell.
- `HTMX` — displays a running app in an **iframe** inside the notebook, either by URL or by rendering an `FT` component directly via a temp route.
- `htmx_config_port` — injects a JS snippet that rewrites relative htmx request paths to include the correct host/port, so htmx calls from the iframe hit the right server.

**WebSocket helper**
- `ws_client` — sets up a WebSocket-connected component displayed in an iframe, with a `send()` helper for pushing updates from Python to the browser.

In short: it bridges the gap between a notebook kernel and a live FastHTML web app, handling server lifecycle, port routing, and in-notebook rendering.

## helpers

In [ ]:
#| exporti
async def maybe_await(o): return await o if inspect.isawaitable(o) else o

In [ ]:
#| export
def kill_port(port): subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

In [ ]:
kill_port(8000)
test_is(is_port_free(8000), True)

In [ ]:
#| export
def connect_host(host): return 'localhost' if host in ('0.0.0.0', '::') else host

def is_serving(srv: JupyUvi, timeout=1):
    "Check if a JupyUvi instance is live and serving requests"
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        return httpx.get(f"http://{connect_host(srv.host)}:{srv.port}/", timeout=timeout).status_code < 500
    except Exception: return False

async def is_serving_a(srv: JupyUvi, timeout=1):
    "Check if a JupyUvi instance is live and serving requests"
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        async with httpx.AsyncClient(timeout=timeout) as cli: return (await cli.get(f"http://{connect_host(srv.host)}:{srv.port}/")).status_code < 500
    except Exception: return False

In [ ]:
test_eq(is_serving(None), False)
test_eq(await is_serving_a(None), False)

In [ ]:
#| export
def wrap_endp(response):
    "If `response` is not callable, wrap a str/FT/sequence str|FT as a async endpoint"
    if callable(response) and not isinstance(response, FT) and not hasattr(response, '__ft__'): return response
    async def endpoint(request): return response
    return endpoint

In [ ]:
#| export
def find_server(port=8000, all=False):
    "Scan IPython user_ns for running JupyUvi instance(s), optionally on `port`"
    import IPython
    if all: port = None
    ns = IPython.get_ipython().user_ns
    res = [(v,k) for k,v in ns.items() if isinstance(v, JupyUvi) and (port is None or v.port == port) and v.server and v.server.started and not v.server.should_exit]
    return res if all else res[0] if res else (None,None)

In [ ]:
test_eq(find_server(), (None,None))
test_eq(find_server(None, all=True), [])

In [ ]:
srv = JupyUvi(app := FastHTML(), port=6551)

In [ ]:
srv2,_ = find_server(6551)
test_is(srv2, srv)

In [ ]:
test_is(find_server(6551)[0], srv)
test_eq(find_server()[0], None)
test_eq(find_server(None, all=True), find_server(all=True))
test_eq(set([_[1] for _ in find_server(all=True)]), {'srv', 'srv2'})

In [ ]:
srv.stop()
await asyncio.sleep(0.2)
test_eq(find_server(None, all=True), [])

In [ ]:
import os
os.environ['PUBLIC_DOMAINS']

'{"6000": "atun-zahara-gran-buen7o.solveit.pub", "8000": "known-vent-santa-eras34.solveit.pub", "8001": "zen-koala-sings-pww6ud.solveit.pub", "8002": "peace-garden-wander-bg59t.solveit.pub", "8003": "sweet-deer-drifts-ez9z8c.solveit.pub"}'

In [ ]:
#| export
_open_pub_ports = tuple(json.loads(os.environ["PUBLIC_DOMAINS"]).keys())

In [ ]:
_open_pub_ports

('6000', '8000', '8001', '8002', '8003')

In [ ]:
#| export
def pub_ports():
    "Return public ports reserved by Solveit"
    return set(map(int, json.loads(os.environ.get('PUBLIC_DOMAINS', '{}'))))

def free_port(start=6550, skip=(), host='localhost'):
    "Find first free port >= `start`, excluding `skip`"
    skip = set(skip)
    return first(p for p in range(start, 65536) if p not in skip and is_port_free(p, host))

In [ ]:
p = free_port(6551, skip=pub_ports())
srv = JupyUvi(FastHTML(), port=p)
test_eq(is_serving(srv), True)
test_eq(await is_serving_a(srv), True)
srv.stop()
await asyncio.sleep(0.2)
test_eq(is_serving(srv), False)
test_eq(await is_serving_a(srv), False)

In [ ]:
test_eq(free_port(6550), 6550)
test_eq(free_port(6550, skip=[6550,6551]), 6552)
test_eq(free_port(5001, skip=[5001,8000]), 5002)

## server

In [ ]:
#| export
daisyui_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
)
threejs_hdrs = (
    Script('{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.165.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.165.0/examples/jsm/"}}', type='importmap'),
)

In [ ]:
#| export
def mk_app(app_kw=None):
    "Create default FastHTML app"
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs)
    return FastHTML(**app_kw)

In [ ]:
test_eq(pub_ports(), set(map(int, json.loads(os.environ.get('PUBLIC_DOMAINS', '{}')))))
test_eq(free_port(6551, skip=pub_ports()), 6551)

a = mk_app({})
test_eq(len(a.hdrs), 6)
test_eq(all(h not in a.hdrs for h in daisyui_hdrs), True)

In [ ]:
#| export
def get_preview(app, port=8000, host='localhost'): return partial(HTMX, app=app, port=port, host=host)

In [ ]:
#| export
def stop_server(port=8000):
    "Stop the running server on `port` and clear all namespace aliases to it"
    srv,_ = find_server(port)
    if not srv: return
    if is_serving(srv): srv.stop()
    for k,v in list(get_ipython().user_ns.items()):
        if v is srv: set_var(k, None)

async def stop_server_a(port=8000):
    "Stop the running server on `port` and clear all namespace aliases to it"
    srv,_ = find_server(port)
    if not srv: return
    if await is_serving_a(srv): await maybe_await(srv.stop())
    for k,v in list(get_ipython().user_ns.items()):
        if v is srv: set_var(k, None)

In [ ]:
p = free_port(6551, skip=pub_ports())
tst_srv = JupyUvi(mk_app({}), port=p)
test_is(find_server(p)[0], tst_srv)
test_eq(is_serving(tst_srv), True)
stop_server(p)
await asyncio.sleep(0.2)
test_eq(find_server(p), (None,None))
test_eq(tst_srv, None)

In [ ]:
tst_srv_a = JupyUviAsync(mk_app({}), port=p)
await tst_srv_a.start()
test_is(find_server(p)[0], tst_srv_a)
test_eq(await is_serving_a(tst_srv_a), True)
await stop_server_a(p)
test_eq(find_server(p), (None,None))
test_eq(tst_srv_a, None)

In [ ]:
#| export
def srv_bundle(srv): return srv,srv.app,srv.app.route,get_preview(srv.app, srv.port)

def _server_target(srv=None, port=8000, reuse=True, host='localhost'):
    own = False
    if srv: return srv,port,own
    if port is None: port = free_port(6551, skip=pub_ports(), host=host)
    found,_ = find_server(port)
    if found:
        if not reuse: raise RuntimeError(f"Server already running on port {port}")
        return found,port,own
    if not is_port_free(port): raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
    return None,port,True

def _ensure_server(srv=None, port=8000, app=None, app_kw=None, reuse=True, **jupyuvi_kw):
    srv,port,own = _server_target(srv, port, reuse, jupyuvi_kw.get('host', 'localhost'))
    if srv:
        if not is_serving(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return (*srv_bundle(srv), own)
    srv = JupyUvi(app or mk_app(app_kw), port=port, **jupyuvi_kw)
    return (*srv_bundle(srv), own)

async def _ensure_server_a(srv=None, port=8000, app=None, app_kw=None, reuse=True, **jupyuvi_kw):
    srv,port,own = _server_target(srv, port, reuse, jupyuvi_kw.get('host', 'localhost'))
    if srv:
        if not await is_serving_a(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return (*srv_bundle(srv), own)
    srv = JupyUviAsync(app or mk_app(app_kw), port=port, **jupyuvi_kw)
    await srv.start()
    return (*srv_bundle(srv), own)

In [ ]:
srv, app, _, _, _ = _ensure_server()
test_is(srv is not None, True)

srv2, app2, _, _, _ = await _ensure_server_a()
test_eq((app,srv), (app2,srv2))

In [ ]:
_srv, sym = find_server()
test_eq((_srv, sym), (srv, 'srv'))

In [ ]:
await stop_server_a()

test_is(globals()['srv'], None)
test_is(globals()['srv2'], None)
test_eq(find_server(), (None, None))
test_is(is_serving(srv2), False)

Test explicit-port start/reuse/stop.

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,app,rt,_,_ = _ensure_server(port=port)
test_eq((srv.port, is_serving(srv)), (port, True))

srv2,app2,rt2,_,_ = _ensure_server(port=port)
test_eq((app2, srv2), (app, srv))

await stop_server_a(port)


`port=None` chooses a non-public free port and starts cleanly:

In [ ]:
srv,app,_,p,_ = _ensure_server(port=None)
test_eq(srv.port in pub_ports(), False)
test_eq(srv.port >= 6551, True)
test_eq(p.keywords['port'], srv.port)
test_is(p.keywords['app'], app)

await stop_server_a(srv.port)

Passing the same app is accepted and reused:

In [ ]:
port = free_port(6551, skip=pub_ports())
app = FastHTML()
srv1,app1,_,_,_ = _ensure_server(port=port, app=app)
srv2,app2,_,_,_ = _ensure_server(port=port, app=app)
test_eq((app1,srv1), (app,srv2))

await stop_server_a(port)

In [ ]:
dead,app,_,_,_ = _ensure_server(port=None)
dead_ref = [dead]

dead.stop()
await asyncio.sleep(0.2)

test_eq(is_serving(dead_ref[0]), False)
test_fail(lambda: _ensure_server(srv=dead_ref[0]), contains='not serving', exc=RuntimeError)

dead_ref.clear()

Test the “existing server wins” contract explicitly, including that a passed `app` is ignored when a server already exists on the requested port.

In [ ]:
port = free_port(6551, skip=pub_ports())
app1,app2 = FastHTML(),FastHTML()
srv,app,rt,p,_ = _ensure_server(port=port, app=app1)
srv2,app3,rt2,p2,_ = await _ensure_server_a(port=port, app=app2)

test_is(srv2, srv)
test_is(app3, app1)
test_is(app3, srv.app)
test_eq(p2.keywords['port'], port)

await stop_server_a(port)

In [ ]:
srv,app,_,_,own = _ensure_server()
srv2,app2,_,_,own2 = await _ensure_server_a()

test_eq((own, own2), (True, False))
await stop_server_a(srv.port)

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,app,rt,_,_ = await _ensure_server_a(port=port)

with expect_fail(RuntimeError, 'already running'):
    await _ensure_server_a(port=port, reuse=False)
test_eq(await is_serving_a(srv), True)

await stop_server_a(port)

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,app,rt,p,own = await _ensure_server_a(port=port)
ref = [srv]
await stop_server_a(port)

with expect_fail(RuntimeError, 'not serving'):
    await _ensure_server_a(srv=ref[0])

test_eq(await is_serving_a(ref[0]), False)
ref.clear()

In [ ]:
#| export
def ensure_server(srv=None, port=8000, app=None, app_kw=None, **jupyuvi_kw):
    "Return a live server, served app, route decorator, and preview helper; existing servers win"
    srv,app,rt,p,_ = _ensure_server(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, **jupyuvi_kw)
    return srv,app,rt,p

async def ensure_server_a(srv=None, port=8000, app=None, app_kw=None, **jupyuvi_kw):
    "Return a live async server, served app, route decorator, and preview helper; existing servers win"
    srv,app,rt,p,_ = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, **jupyuvi_kw)
    return srv,app,rt,p

`ensure_server` returns the standard notebook/Solveit serving bundle: a live server, the FastHTML app it serves, the app route decorator, and a preview helper wired to that server.

Existing servers have priority. If a server is passed explicitly, it must be live and is used. Otherwise, an already-running server on the requested port is reused. If no matching server exists, a new one is started, using the provided app if any, otherwise creating a default app.

The returned preview helper always targets the returned server's port.

`stop_server` stops the running server on a port and clears all notebook namespace names pointing at that server.

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,app,rt,p = ensure_server(port=port)

test_eq((srv.port, is_serving(srv)), (port, True))
test_is(app, srv.app)
test_eq(p.keywords['port'], port)
test_is(p.keywords['app'], app)

await stop_server_a(port)

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,app,rt,p = ensure_server(port=port)
srv2,app2,rt2,p2 = ensure_server(port=port)

test_eq((srv2,app2,rt2), (srv,app,rt))
test_eq(p2.keywords['port'], port)

await stop_server_a(port)

## test rig

In [ ]:
#| exporti
def log(msg, *args, **kwargs): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

In [ ]:
log = await logger()

In [ ]:
#| export
class LoggerHandler(logging.Handler):
    def __init__(self, log): self.log = log; super().__init__()
    def emit(self, record):
        try: self.log(f"{(record.levelname+':'):<10}{self.format(record)}")
        except Exception: self.handleError(record)

In [ ]:
got = []
h = LoggerHandler(got.append)
h.emit(logging.LogRecord('x', logging.WARNING, '', 1, 'careful', (), None))
test_eq(got, ['WARNING:  careful'])

In [ ]:
class BadLog:
    def __call__(self, x): raise RuntimeError('boom')

h = LoggerHandler(BadLog())
h.handleError = lambda record: setattr(h, 'handled', True)
h.emit(logging.LogRecord('x', logging.INFO, '', 1, 'msg', (), None))
test_eq(h.handled, True)

In [ ]:
#| export
def setup_uvi_logging(_log=None, level=logging.INFO):
    "Route uvicorn logs to `_log`, returning a cleanup function"
    _log = log if _log is None else _log
    _lh = LoggerHandler(_log)
    names = ('uvicorn', 'uvicorn.error', 'uvicorn.access')
    prev = {}
    for name in names:
        lg = logging.getLogger(name)
        prev[name] = lg.handlers[:],lg.propagate,lg.level,lg.disabled
        lg.handlers,lg.propagate,lg.disabled = [_lh],False,False
        lg.setLevel(level)
    def cleanup():
        for name,(handlers,propagate,level,disabled) in prev.items():
            lg = logging.getLogger(name)
            lg.handlers,lg.propagate,lg.disabled = handlers,propagate,disabled
            lg.setLevel(level)
    return cleanup

Verify all three uvicorn loggers are changed and then restored.

In [ ]:
names = ('uvicorn','uvicorn.error','uvicorn.access')
logs = [logging.getLogger(o) for o in names]
old = [(o.handlers[:],o.propagate,o.level,o.disabled) for o in logs]
cleanup = setup_uvi_logging(lambda x: None, level=logging.WARNING)

test_eq([o.level for o in logs], [logging.WARNING]*3)
test_eq([o.propagate for o in logs], [False]*3)
test_eq([o.disabled for o in logs], [False]*3)
test_eq([len(o.handlers) for o in logs], [1]*3)

cleanup()
test_eq([(o.handlers,o.propagate,o.level,o.disabled) for o in logs], old)

`setup_uvi_logging` routes uvicorn log output to a supplied logger callable, defaulting to this module's `log`.

It configures the uvicorn, uvicorn error, and uvicorn access loggers at the requested level, and returns a cleanup function that restores their previous logging state. Use the cleanup function when the custom routing is no longer needed.

In [ ]:
#| export
@asynccontextmanager
async def serve_app(
    app=None, # FastHTML app to serve; if omitted, create one with `app_kw`
    port=8000, # Port to serve on; `None` chooses a free non-public port
    srv=None, # Existing server to use; must already be live
    reuse=True, # Reuse an existing live server on `port`
    after_start=None, # Optional callback run after server is available
    before_stop=None, # Optional callback run before owned-server teardown
    restore_routes=True, # Restore `app.routes` after the block for owned servers
    timeout=5, # Seconds to wait for owned server port to become free
    app_kw=None, # Keyword args for default app creation
    **jupyuvi_kw):
    "Yield a live JupyUviAsync server for a test block; reuse only when allowed, cleanup only when owned, restore owned routes"
    srv,app,rt,p,own = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=reuse, **jupyuvi_kw)
    routes,err = list(app.routes) if own and restore_routes and hasattr(app, 'routes') else None,None
    try:
        if after_start: await maybe_await(after_start(srv))
        yield srv
    except Exception as e: err = e
    try:
        if before_stop: await maybe_await(before_stop(srv))
    except Exception as e:
        if err is None: err = e
    try:
        if routes is not None: app.routes[:] = routes
    except Exception as e:
        if err is None: err = e
    if own:
        srv.server.should_exit = True
        deadline = (loop := asyncio.get_running_loop()).time() + timeout
        while not is_port_free(srv.port, srv.host):
            if loop.time() > deadline:
                e = TimeoutError(f"Port {srv.host}:{srv.port} still in use after {timeout}s")
                if err is None: err = e
                break
            await asyncio.sleep(0.1)
    if err: raise err

In [ ]:
async with serve_app(port=None) as srv:
    test_eq(await is_serving_a(srv), True)
    test_eq(srv.port >= 6551, True)
    test_eq(srv.port in pub_ports(), False)
test_eq(is_port_free(srv.port), True)

Temporary live server and cleans it up.

In [ ]:
port = free_port(6551, skip=pub_ports())
base_srv,base_app,_,_ = await ensure_server_a(port=port)

async with serve_app(port=port) as srv: test_is(srv, base_srv)

test_eq(await is_serving_a(base_srv), True)
await stop_server_a(port)

Existing servers are reused and not owned.

In [ ]:
port = free_port(6551, skip=pub_ports())

async with serve_app(port=port) as srv:
    n = len(srv.app.routes)
    @srv.app.get('/tmp-route')
    def tmp_route(): return 'ok'

test_eq(len(srv.app.routes), n)
test_eq(is_port_free(port), True)

Owned app routes are restored.

In [ ]:
port = free_port(6551, skip=pub_ports())
err = None

async def bad_before(srv): raise RuntimeError('before boom')

try:
    async with serve_app(port=port, before_stop=bad_before) as srv:
        n = len(srv.app.routes)
        srv.app.get('/tmp-route')(lambda: 'ok')
except RuntimeError as e: err = e

test_eq(str(err), 'before boom')
test_eq(len(srv.app.routes), n)
test_eq(is_port_free(port), True)

INFO:     Waiting for application shutdown.


INFO:     Application shutdown complete.


INFO:     Finished server process [1170]


Cleanup and route restoration still happen when teardown callback fails.

In [ ]:
# Add log_config=None to prevent Uvicorn calling logging.config.dictConfig() on startup
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

INFO:     Started server process [1170]


INFO:     Waiting for application startup.


TRACE:    ASGI [1] Started scope={'type': 'lifespan', 'asgi': {'version': '3.0', 'spec_version': '2.0'}, 'state': {}}


TRACE:    ASGI [1] Receive {'type': 'lifespan.startup'}


TRACE:    ASGI [1] Send {'type': 'lifespan.startup.complete'}


INFO:     Application startup complete.


INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     Shutting down


The `log_config=None` kwarg passed in the test message prevents uvicorn from overwriting the custom `LoggerHandler` logging setup with its own `dictConfig` on startup.

In [ ]:
setup_uvi_logging()
await log.show(True)

[17:59:14.966] INFO:     Started server process [1170]
[17:59:15.020] INFO:     Waiting for application startup.
[17:59:15.070] TRACE:    ASGI [1] Started scope={'type': 'lifespan', 'asgi': {'version': '3.0', 'spec_version': '2.0'}, 'state': {}}
[17:59:15.121] TRACE:    ASGI [1] Receive {'type': 'lifespan.startup'}
[17:59:15.174] TRACE:    ASGI [1] Send {'type': 'lifespan.startup.complete'}
[17:59:15.221] INFO:     Application startup complete.
[17:59:15.276] INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
[17:59:15.450] INFO:     Shutting down
[17:59:15.605] INFO:     Waiting for application shutdown.
[17:59:15.657] TRACE:    ASGI [1] Receive {'type': 'lifespan.shutdown'}
[17:59:15.710] TRACE:    ASGI [1] Send {'type': 'lifespan.shutdown.complete'}
[17:59:15.764] TRACE:    ASGI [1] Completed
[17:59:15.821] INFO:     Application shutdown complete.
[17:59:15.872] INFO:     Finished server process [1170]


In [ ]:
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

`serve_app` is an async context manager for temporarily making sure a FastHTML app is being served during a block of code, with predictable cleanup semantics.

Its contract is:

- On entry, it yields a live `JupyUviAsync` server.
- You may give it:
  - an existing `srv`, which must already be live
  - a `port`, where `None` means “choose a free non-public port”
  - an `app`, or `app_kw` for creating a default app
  - `reuse=True/False` to control whether an existing server on that port may be reused
- If it creates/owns the server, it cleans it up when the block exits.
- If it reuses an existing server, it leaves that server running after the block.
- For owned servers, it can restore `app.routes` after the block, so temporary routes added inside the context don’t leak out.
- It runs optional lifecycle callbacks:
  - `after_start(srv)` after the server is available
  - `before_stop(srv)` before owned-server teardown
- It propagates errors from the block or callbacks, while still attempting cleanup.
- It waits for an owned server’s port to become free on exit, up to `timeout`.
- It returns no app/route/preview bundle directly — just the live server — so inside the block you use `srv.app`, `srv.port`, etc.

So conceptually:

```python
async with serve_app(port=None) as srv:
    ...
```

means: “Give me a live temporary server for this block, and if you started it for me, clean it up afterward.”

# export -

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose" markdown="1">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()